# External Validation: KP Models on All of Us Data

Validate Kaiser Permanente (KP) trained models on All of Us (AoU) cohort data

**Workflow:**
1. Load All of Us cohort data (outcomes, medical history, demographics, labs, vitals)
2. Load pre-trained KP model (XGBoost .json file)
3. Choose to Fine-tune KP model on subset of AoU data
4. Evaluate on remaining AoU data
5. Generate overall and subgroup performance metrics with bootstrap confidence intervals

File is commented out because it relies on external models built on the KPNC dataset. 


# Setup

In [ ]:
'''
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, auc, precision_recall_curve,
    accuracy_score, confusion_matrix, 
    average_precision_score, balanced_accuracy_score, 
    precision_score, recall_score,
    brier_score_loss, log_loss, roc_curve
)
from sklearn import utils

from xgboost import XGBClassifier
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict
'''

In [ ]:
'''
# Environment setup
version = %env WORKSPACE_CDR
my_bucket = os.getenv('WORKSPACE_BUCKET')

print(f"Workspace CDR: {version}")
print(f"Workspace bucket: {my_bucket}")
'''

# Data Loading

Load All of Us cohort data components:
- **Outcomes**: Pregnancy complications (PE, GDM, GHTN, etc.)
- **Medical History**: Pre-existing conditions and comorbidities
- **Demographics**: Age, race, ethnicity, birth date
- **Labs**: Laboratory measurements (pp_ and ep_ time periods)
- **Vitals**: Blood pressure, heart rate (pp_ and ep_ time periods)

In [ ]:
'''
print("Loading All of Us cohort data...\n")

# Load data components
df_outcomes = pd.read_pickle('pregnancy_cohort_outcomes_date.pkl')
df_hx = pd.read_pickle('pregnancy_cohort_hx_date.pkl')
df_demo = pd.read_csv('pregnancy_demographics_date.csv')
df_labs = pd.read_csv('labs_hold_24w_date.csv')
df_vitals = pd.read_csv('vitals_hold_24w_date.csv')

print(f"Outcomes shape: {df_outcomes.shape}")
print(f"Medical history shape: {df_hx.shape}")
print(f"Demographics shape: {df_demo.shape}")
print(f"Labs shape: {df_labs.shape}")
print(f"Vitals shape: {df_vitals.shape}")

print(f"\nOutcome columns: {df_outcomes.columns.tolist()}")
'''

## Data Merging

Merge all data components on episode ID

In [ ]:
'''
print("Merging datasets...\n")

# Merge outcomes with medical history
df = df_outcomes.merge(df_hx, how='left', on='id')

# Merge with demographics
df = df.merge(df_demo, how='left', on='id')

# Merge labs and vitals first
labs_vitals = df_vitals.merge(df_labs, how='outer', on='id')

# Then merge with main dataframe (inner join to keep only episodes with vital/lab data)
df = df.merge(labs_vitals, how='inner', on='id')

print(f"Merged dataset shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")
'''

# Feature Engineering

Create derived features needed for model evaluation

In [ ]:
'''
print("Engineering features...\n")

# Convert censor_date to timezone-naive datetime
for i in range(len(df)):
    df.loc[i, 'censor_date_x'] = pd.to_datetime(df.loc[i, 'censor_date_x']).replace(tzinfo=None)

# Calculate maternal age at conception
df['mage'] = (np.floor(
    (pd.to_datetime(df['censor_date_x']) - pd.to_datetime(df['birth_date'])).dt.days / 365
)).astype(int)


# Rename outcome columns for consistency
df.rename(columns={
    'Gestational Diabetes': 'GDM_1', 
    'Pre-eclampsia': 'PE', 
    'Gestational_Hypertension': 'GHTN',
    'Eclampsia': 'E'
}, inplace=True)


# Calculate parity (whether patient has had previous pregnancies)
df['parity'] = (df.groupby('person_id_x')['censor_date_x'].transform('nunique') > 1).astype(int)


# Clean BMI outliers
df['PPREG_BMI'] = df['PPREG_BMI'].mask(
    (df['PPREG_BMI'] < 10) | (df['PPREG_BMI'] > 100), 
    np.nan
)
'''

## Race/Ethnicity Categories

In [ ]:
'''
# Create race/ethnicity categories
df['Hispanic'] = df['ethnicity'] == 'Hispanic or Latino'
df['NH_White'] = (df['ethnicity'] != 'Hispanic or Latino') & (df['race'] == 'White')
df['NH_Black'] = (df['ethnicity'] != 'Hispanic or Latino') & (df['race'] == 'Black or African American')
df['NH_Asian'] = (df['ethnicity'] != 'Hispanic or Latino') & (
    df['race'].isin(['Asian', 'Native Hawaiian or Other Pacific Islander'])
)
df['Other'] = (df['ethnicity'] != 'Hispanic or Latino') & (
    ~df['race'].isin(['White', 'Black or African American', 'Asian', 'Native Hawaiian or Other Pacific Islander'])
)

# Convert to integer
df[['Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other']] = \
    df[['Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other']].astype(int)

print(" Race/ethnicity categories created")

# Distribution
print("\nRace/Ethnicity Distribution:")
for race in ['Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other']:
    n = df[race].sum()
    pct = n / len(df) * 100
    print(f"  {race}: {n:,} ({pct:.1f}%)")
'''

## Remove Pregnancies without Data

In [ ]:
'''
# Remove rows with no vital/lab data at all
before_count = len(df)
df = df.dropna(subset=df.filter(regex=r"^(ep|pp)").columns, how="all")
after_count = len(df)

print(f"Removed {before_count - after_count:,} pregnancies with no vital/lab data")
print(f"Final dataset shape: {df.shape}")
'''

# KP Feature List Definition

**IMPORTANT:** Features must match the Kaiser Permanente model's expected input format and order


In [ ]:

# KP model feature list - MUST be in exact order expected by model

# #list below is for Pre-eclampsia
# 
# feats_to_train = [
#     # Pre-pregnancy labs (avg, min, max, std)
#     'pp_ALT_avg', 'pp_AST_avg', 'pp_CREATININE_avg', 'pp_GLU_F_avg', 'pp_HGB_avg', 'pp_RBC_avg', 
#     'pp_WBC_avg', 'pp_ANC_avg', 'pp_CHLORIDE_avg', 'pp_GLU_RAN_avg', 'pp_CALCIUM_avg', 
#     'pp_BUN_avg', 'pp_GTT75_2_avg', 'pp_WBC_COR_avg', 'pp_GTTPM_1_avg', 'pp_GTT_2_avg', 
#     'pp_GTT_1_avg', 'pp_GTTPM_2_avg', 'pp_GTT_3_avg',
    
#     'pp_ALT_min', 'pp_AST_min', 'pp_CREATININE_min', 'pp_GLU_F_min', 'pp_HGB_min', 
#     'pp_RBC_min', 'pp_WBC_min', 'pp_ANC_min', 'pp_CHLORIDE_min', 'pp_GLU_RAN_min', 
#     'pp_CALCIUM_min', 'pp_BUN_min', 'pp_GTT75_2_min', 'pp_WBC_COR_min', 'pp_GTTPM_1_min', 
#     'pp_GTT_2_min', 'pp_GTT_1_min', 'pp_GTTPM_2_min', 'pp_GTT_3_min',
    
#     'pp_ALT_max', 'pp_AST_max', 'pp_CREATININE_max', 'pp_GLU_F_max', 'pp_HGB_max', 
#     'pp_RBC_max', 'pp_WBC_max', 'pp_ANC_max', 'pp_CHLORIDE_max', 'pp_GLU_RAN_max', 
#     'pp_CALCIUM_max', 'pp_BUN_max', 'pp_GTT75_2_max', 'pp_WBC_COR_max', 'pp_GTTPM_1_max', 
#     'pp_GTT_2_max', 'pp_GTT_1_max', 'pp_GTTPM_2_max', 'pp_GTT_3_max',
    
#     'pp_ALT_std', 'pp_AST_std', 'pp_CREATININE_std', 'pp_GLU_F_std', 'pp_HGB_std', 
#     'pp_RBC_std', 'pp_WBC_std', 'pp_ANC_std', 'pp_CHLORIDE_std', 'pp_GLU_RAN_std', 
#     'pp_CALCIUM_std', 'pp_BUN_std', 'pp_GTT75_2_std', 'pp_WBC_COR_std', 'pp_GTTPM_1_std', 
#     'pp_GTT_2_std', 'pp_GTT_1_std', 'pp_GTTPM_2_std', 'pp_GTT_3_std',
    
#     # Early pregnancy labs (avg, min, max, std)
#     'ep_HGB_avg', 'ep_RBC_avg', 'ep_WBC_avg', 'ep_GLU_RAN_avg', 'ep_GTTPM_1_avg', 
#     'ep_AST_avg', 'ep_ANC_avg', 'ep_ALT_avg', 'ep_CREATININE_avg', 'ep_GLU_F_avg', 
#     'ep_BUN_avg', 'ep_CHLORIDE_avg', 'ep_GTT_1_avg', 'ep_GTT_2_avg', 'ep_GTT_3_avg', 
#     'ep_CALCIUM_avg', 'ep_GTT75_2_avg', 'ep_WBC_COR_avg', 'ep_GTTPM_2_avg',
    
#     'ep_HGB_min', 'ep_RBC_min', 'ep_WBC_min', 'ep_GLU_RAN_min', 'ep_GTTPM_1_min', 
#     'ep_AST_min', 'ep_ANC_min', 'ep_ALT_min', 'ep_CREATININE_min', 'ep_GLU_F_min', 
#     'ep_BUN_min', 'ep_CHLORIDE_min', 'ep_GTT_1_min', 'ep_GTT_2_min', 'ep_GTT_3_min', 
#     'ep_CALCIUM_min', 'ep_GTT75_2_min', 'ep_WBC_COR_min', 'ep_GTTPM_2_min',
    
#     'ep_HGB_max', 'ep_RBC_max', 'ep_WBC_max', 'ep_GLU_RAN_max', 'ep_GTTPM_1_max', 
#     'ep_AST_max', 'ep_ANC_max', 'ep_ALT_max', 'ep_CREATININE_max', 'ep_GLU_F_max', 
#     'ep_BUN_max', 'ep_CHLORIDE_max', 'ep_GTT_1_max', 'ep_GTT_2_max', 'ep_GTT_3_max', 
#     'ep_CALCIUM_max', 'ep_GTT75_2_max', 'ep_WBC_COR_max', 'ep_GTTPM_2_max',
    
#     'ep_HGB_std', 'ep_RBC_std', 'ep_WBC_std', 'ep_GLU_RAN_std', 'ep_GTTPM_1_std', 
#     'ep_AST_std', 'ep_ANC_std', 'ep_ALT_std', 'ep_CREATININE_std', 'ep_GLU_F_std', 
#     'ep_BUN_std', 'ep_CHLORIDE_std', 'ep_GTT_1_std', 'ep_GTT_2_std', 'ep_GTT_3_std', 
#     'ep_CALCIUM_std',
    
#     # Vitals (pre-pregnancy and early pregnancy)
#     'pp_hr_avg', 'pp_sbp_avg', 'pp_dbp_avg', 'pp_hr_min', 'pp_sbp_min', 'pp_dbp_min', 
#     'pp_hr_max', 'pp_sbp_max', 'pp_dbp_max', 'pp_hr_std', 'pp_sbp_std', 'pp_dbp_std',
    
#     'ep_hr_avg', 'ep_sbp_avg', 'ep_dbp_avg', 'ep_hr_min', 'ep_sbp_min', 'ep_dbp_min', 
#     'ep_hr_max', 'ep_sbp_max', 'ep_dbp_max', 'ep_hr_std', 'ep_sbp_std', 'ep_dbp_std',
    
#     # Medical history
#     'familyhx_dm_ever', 'dep_priorlmp', 'smk_prior', 'alc_prior', 'pcos', 'hxpe_e', 
#     'hxghtn', 'hxgdm', 'hxabrt', 'mage', 'PPREG_BMI', 'parity', 'cancer_prepreg', 
#     'ga_office1st', 'ins_medi',
    
#     # Comorbidities
#     'chest_pain', 'uti', 'hyperkeratosis', 'carp_tunnel', 'hypercholes', 'cyst_ovary', 
#     'wart', 'pain_rightquad', 'viraldisease_preg', 'resp_infection', 'pain_neck', 
#     'pain_joint', 'pain_chronic', 'acne', 'breast_lump', 'pain_limb', 'ab_pain', 
#     'ten_headache', 'headache', 'viraldisease', 'dm', 'chtn_prepreg',
    
#     # Race/ethnicity
#     'Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other'
# ]
#  

'''
print(f"\nFeature categories:")
print(f"  - Pre-pregnancy labs: {sum(1 for f in feats_to_train if f.startswith('pp_') and not f.startswith(('pp_hr', 'pp_sbp', 'pp_dbp')))}")
print(f"  - Early pregnancy labs: {sum(1 for f in feats_to_train if f.startswith('ep_') and not f.startswith(('ep_hr', 'ep_sbp', 'ep_dbp')))}")
print(f"  - Vitals: {sum(1 for f in feats_to_train if any(f.startswith(p) for p in ['pp_hr', 'pp_sbp', 'pp_dbp', 'ep_hr', 'ep_sbp', 'ep_dbp']))}")
print(f"  - Medical history/demographics: {sum(1 for f in feats_to_train if not f.startswith(('pp_', 'ep_')))}")
'''

# Outcome Selection

Select the outcome to evaluate (change this variable to test different outcomes)

In [ ]:
'''
# SELECT OUTCOME HERE
outcome = 'Premature'
# Options: 'Pre-eclampsia_12', 'Gestational Diabetes_12', 'Gestational_Hypertension_12'
#'Stillbirth', 'Premature', 'Placental_Abruption'

print(f"SELECTED OUTCOME: {outcome}")
print(f"\nOutcome prevalence:")
n_positive = df[outcome].sum()
n_total = len(df)
print(f"  Positive cases: {n_positive:,}")
print(f"  Total cases: {n_total:,}")
print(f"  Prevalence: {n_positive/n_total*100:.2f}%")

# Set outcome variable
y = df[outcome]
'''

In [ ]:
'''
# Create model features list (add person_id for splitting)
model_features = feats_to_train.copy()
model_features.append('person_id_x')

# Select columns that exist in AoU data
df_external = df[[col for col in model_features if col in df.columns]].copy()

print(f"Features present in AoU data: {len(df_external.columns)}")
print(f"Dataset shape: {df_external.shape}")

# Add NaN columns for features that don't exist in AoU
missing_features = [col for col in model_features if col not in df_external.columns]
print(f"\nFeatures missing from AoU (will be filled with NaN): {len(missing_features)}")
if missing_features:
    print("Missing features:")
    for feat in missing_features:
        print(f"  - {feat}")
        df_external[feat] = np.nan

# Reorder to match expected feature order
df_external = df_external[model_features]

print(f"\nFinal external validation dataset shape: {df_external.shape}")
'''

## Functions for evaluation

In [ ]:
'''
def ex_validate(df_external, y, finetune, percent, model_file):
    """
    External validation function
    
    Parameters:
    -----------
    df_external : DataFrame
        Feature matrix with person_id_x column
    y : Series
        Outcome labels
    finetune : bool
        Whether to fine-tune on subset of external data
    percent : float
        Fraction of data to use for fine-tuning (e.g., 0.2 = 20%)
    model_file : str
        Path to KP model .json file
    
    Returns:
    --------
    n : int
        Number of positive cases in evaluation set
    tot : int
        Total number of cases in evaluation set
    y_eval : Series
        Outcome labels for evaluation set
    df_external_scaled : DataFrame
        Scaled feature matrix for evaluation
    model : XGBClassifier
        Loaded (and possibly fine-tuned) model
    """
    # Load pre-trained KP model
    model = XGBClassifier()
    model.load_model(model_file)
    print(f" Loaded KP model from: {model_file}")
    
    scaler = MinMaxScaler()
    
    if finetune:
        print("\n=== FINE-TUNING MODE ===")
        
        # Split by person_id to avoid data leakage
        unique_ids = df_external['person_id_x'].unique()
        finetune_ids, remaining_ids = train_test_split(
            unique_ids, test_size=1 - percent, random_state=0
        )
        
        # Split data
        df_finetune = df_external[df_external['person_id_x'].isin(finetune_ids)]
        df_eval = df_external[df_external['person_id_x'].isin(remaining_ids)]
        
        # Remove person_id before scaling
        df_finetune = df_finetune.drop(columns='person_id_x')
        df_eval = df_eval.drop(columns='person_id_x')
        
        print(f"  Fine-tune set: {df_finetune.shape[0]:,} pregnancies")
        print(f"  Evaluation set: {df_eval.shape[0]:,} pregnancies")
        
        # Get corresponding labels
        y_finetune = y.loc[df_finetune.index]
        y_eval = y.loc[df_eval.index]
        
        print(f"  Fine-tune positive cases: {y_finetune.sum():,} ({y_finetune.sum()/len(y_finetune)*100:.1f}%)")
        print(f"  Eval positive cases: {y_eval.sum():,} ({y_eval.sum()/len(y_eval)*100:.1f}%)")
        
        # Fit scaler on fine-tune set only
        df_finetune_scaled = pd.DataFrame(
            scaler.fit_transform(df_finetune),
            columns=df_finetune.columns,
            index=df_finetune.index
        )
        
        # Apply scaler to evaluation set
        df_eval_scaled = pd.DataFrame(
            scaler.transform(df_eval),
            columns=df_eval.columns,
            index=df_eval.index
        )
        
        print("\n  Fine-tuning model on AoU data.")
        # Fine-tune model on scaled fine-tune data
        model.set_params(learning_rate=0.1)
        model.fit(df_finetune_scaled, y_finetune, xgb_model=model.get_booster())
        
        # Return evaluation set as scaled version
        df_external_scaled = df_eval_scaled
        
    else:
        print("\n=== NO FINE-TUNING MODE ===")
        print("  Using pre-trained KP model as-is")
        
        # Remove person_id
        df_external = df_external.drop(columns='person_id_x')
        
        # Fit scaler and transform full external data
        df_external_scaled = pd.DataFrame(
            scaler.fit_transform(df_external),
            columns=df_external.columns,
            index=df_external.index
        )
        
        y_eval = y
    
    n = y_eval.sum()
    tot = len(y_eval)
    
    return n, tot, y_eval, df_external_scaled, model
'''

In [ ]:
'''
def cis(X, y, model, meta=None, race_cols=None, age_col=None, age_bins=None, 
        parity_col=None, parity_bins=None, n_boot=500):
    """
    Calculate bootstrap confidence intervals for model performance
    
    Parameters:
    -----------
    X : DataFrame
        Feature matrix
    y : Series
        True labels
    model : XGBClassifier
        Trained model
    meta : DataFrame, optional
        Metadata for subgroup analysis (must include race, age, parity columns)
    race_cols : list, optional
        List of race/ethnicity column names
    age_col : str, optional
        Age column name
    age_bins : list of tuples, optional
        Age bin ranges [(low, high), ...]
    parity_col : str, optional
        Parity column name
    parity_bins : list, optional
        Parity values to analyze
    n_boot : int, optional
        Number of bootstrap iterations (default: 500)
    """
    aucs, prcs = [], []
    
    # Store subgroup metrics
    race_auc_dict = defaultdict(list)
    age_auc_dict = defaultdict(list)
    parity_auc_dict = defaultdict(list)
    
    # Print subgroup sample sizes
    if meta is not None:
        meta_test = meta.loc[X.index]
        
        if race_cols:
            print("\nRace/Ethnicity Subgroup Sizes (positive cases):")
            for race in race_cols:
                mask = (meta_test[race] == 1)
                n_positive = y[mask].sum()
                n_total = mask.sum()
                print(f"  {race}: {n_positive} / {n_total} ({n_positive/n_total*100:.1f}%)")
        
        if age_col and age_bins:
            print("\nAge Subgroup Sizes (positive cases):")
            for (lo, hi) in age_bins:
                label = f"{lo}-{hi}"
                mask = (meta_test[age_col] >= lo) & (meta_test[age_col] < hi)
                n_positive = y[mask].sum()
                n_total = mask.sum()
                print(f"  {label}: {n_positive} / {n_total} ({n_positive/n_total*100:.1f}%)")
        
        if parity_col and parity_bins:
            print("\nParity Subgroup Sizes (positive cases):")
            for i in parity_bins:
                mask = (meta_test[parity_col] == i)
                n_positive = y[mask].sum()
                n_total = mask.sum()
                label = "Nulliparous" if i == 0 else "Multiparous"
                print(f"  {label} (parity={i}): {n_positive} / {n_total} ({n_positive/n_total*100:.1f}%)")
    
    print(f"\nRunning {n_boot} bootstrap iterations...")
    
    # Bootstrap iterations
    for i in range(n_boot):
        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{n_boot}")
        
        # Resample with replacement
        X_test, y_test = utils.resample(X, y, replace=True)
        
        # Generate predictions
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
        probs = np.clip(probs, 0, 1)
        
        # Calculate overall metrics
        precision, recall, _ = precision_recall_curve(y_test, probs)
        aucs.append(round(roc_auc_score(y_test, probs), 3))
        prcs.append(round(auc(recall, precision), 3))
        
        # Subgroup analysis
        if meta is not None:
            meta_test = meta.loc[X_test.index]
            
            # Race subgroups
            if race_cols:
                for race in race_cols:
                    mask = (meta_test[race] == 1)
                    if mask.sum() >= 10:
                        try:
                            rauc = roc_auc_score(y_test[mask], probs[mask])
                            race_auc_dict[race].append(round(rauc, 3))
                        except ValueError:
                            pass
            
            # Age subgroups
            if age_col and age_bins:
                for (lo, hi) in age_bins:
                    label = f"{lo}-{hi}"
                    mask = (meta_test[age_col] >= lo) & (meta_test[age_col] < hi)
                    if mask.sum() >= 10:
                        try:
                            aauc = roc_auc_score(y_test[mask], probs[mask])
                            age_auc_dict[label].append(round(aauc, 3))
                        except ValueError:
                            pass
            
            # Parity subgroups
            if parity_col and parity_bins:
                for i in parity_bins:
                    mask = (meta_test[parity_col] == i)
                    label = str(i)
                    if mask.sum() >= 10:
                        try:
                            aauc = roc_auc_score(y_test[mask], probs[mask])
                            parity_auc_dict[label].append(round(aauc, 3))
                        except ValueError:
                            pass
    
    # Print results
    print("\n" + "="*70)
    print("BOOTSTRAP CONFIDENCE INTERVALS (95%)")
    print("="*70)
    
    print("\nOverall Performance:")
    print(f"  ROC AUC: {np.mean(aucs):.3f} {np.round(np.percentile(aucs, [2.5, 97.5]), 3)}")
    print(f"  PR AUC:  {np.mean(prcs):.3f} {np.round(np.percentile(prcs, [2.5, 97.5]), 3)}")
    
    # Subgroup results
    if race_auc_dict:
        print("\nRace/Ethnicity Subgroup ROC AUCs:")
        for race, scores in race_auc_dict.items():
            if scores:
                ci = np.percentile(scores, [2.5, 97.5])
                print(f"  {race:15s}: {np.mean(scores):.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")
    
    if age_auc_dict:
        print("\nAge Subgroup ROC AUCs:")
        for label, scores in age_auc_dict.items():
            if scores:
                ci = np.percentile(scores, [2.5, 97.5])
                print(f"  {label:15s}: {np.mean(scores):.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")
    
    if parity_auc_dict:
        print("\nParity Subgroup ROC AUCs:")
        for label, scores in parity_auc_dict.items():
            if scores:
                ci = np.percentile(scores, [2.5, 97.5])
                parity_name = "Nulliparous" if label == "0" else "Multiparous"
                print(f"  {parity_name:15s}: {np.mean(scores):.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")
    
    print("\n" + "="*70)
'''

# Run External Val

In [ ]:
'''
# VALIDATION SETTINGS
FINETUNE = True  # Set to False to skip fine-tuning
FINETUNE_PERCENT = 0.5 # Use 20% of data for fine-tuning
MODEL_FILE = 'Preterm_24w_date.json'  # KP model file to load

print("EXTERNAL VALIDATION: KP MODEL ON ALL OF US DATA")
print(f"\nModel: {MODEL_FILE}")
print(f"Outcome: {outcome}")
print(f"Fine-tuning: {'YES' if FINETUNE else 'NO'}")
if FINETUNE:
    print(f"Fine-tune fraction: {FINETUNE_PERCENT*100:.0f}%")
print()

# Run validation
n, tot, targets, x_eval, model = ex_validate(
    df_external, 
    y, 
    FINETUNE, 
    FINETUNE_PERCENT, 
    MODEL_FILE
)

print("EVALUATION SET SUMMARY")
print(f"Total pregnancies: {tot:,}")
print(f"Positive cases: {n:,} ({n/tot*100:.2f}%)")
print(f"Negative cases: {tot-n:,} ({(tot-n)/tot*100:.2f}%)")
'''

In [ ]:
'''
# Generate predictions
preds = model.predict(x_eval)
probs = model.predict_proba(x_eval)[:, 1]
probs = np.clip(probs, 0, 1)

# Calculate metrics
precision, recall, _ = precision_recall_curve(targets, probs)
roc_auc = roc_auc_score(targets, probs)
pr_auc = auc(recall, precision)

print("\n" + "="*70)
print("OVERALL PERFORMANCE METRICS")
print("="*70)
print(f"ROC AUC:  {roc_auc:.3f}")
print(f"PR AUC:   {pr_auc:.3f}")
print(f"\nPositive cases: {n} / {tot} ({n/tot*100:.2f}%)")
'''

## Subgroup Analysis

In [ ]:
'''
# Define subgroups
race_cols = ['Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other']
age_bins = [(0, 25), (25, 30), (30, 35), (35, 100)]
parity_bins = [0, 1]

# Run bootstrap analysis
cis(
    x_eval, 
    targets, 
    model,
    meta=df_external, 
    race_cols=race_cols,
    age_col='mage',
    age_bins=age_bins,
    parity_col='parity',
    parity_bins=parity_bins,
    n_boot=500
)
'''